# Sistema de Gestión de Inventario con MCP Remoto (HTTP)

## Descripción

En este ejercicio, construirás un **sistema de agentes con un servidor MCP remoto auto-hospedado**, que expone herramientas HTTP reutilizables para analizar inventario y ventas semanales en un centro de distribución.

### Arquitectura MCP Remoto
```
Pregunta sobre inventario
    ↓
[InventoryAgent]
    ↓
[MCPStreamableHTTPTool] → http://<codespaces-url>:8000
    ↓
Servidor Python (remote_mcp_server.py)
    ↓
[Herramientas del servidor]
    ├→ get_inventory_levels()
    └→ get_weekly_sales()
    ↓
Análisis y respuesta
```

### Ventajas de MCP Remoto
- **Centralización**: Las herramientas se pueden alojar en un único servidor y ser consumidas por múltiples agentes.
- **Escalabilidad**: El servidor puede escalar de forma independiente a los agentes.
- **Accesibilidad**: Cualquier agente con acceso a la red puede utilizar las herramientas.

### 2. Ejecutar el Servidor en GitHub Codespaces

**Pasos:**
1. Abre una nueva terminal en GitHub Codespaces (`Terminal > New Terminal`).
2. Ejecuta el servidor con el siguiente comando: `python remote_mcp_server.py`
3. Codespaces detectará automáticamente que el puerto `8000` está siendo utilizado y te mostrará una notificación para hacer el reenvío de puertos (port forwarding).
4. Haz clic en **Open in Browser** o copia la URL proporcionada para acceder al servidor MCP.
5. En el panel **Ports**, cambia manualmente la visibilidad del puerto reenviado de **Private** a **Public**.
6. Pega la URL pública resultante en la variable `CODESPACES_FORWARDED_URL` de la celda de código más abajo.

### 3. Configurar la URL expuesta por Codespaces

Define la URL pública reenviada para reutilizarla en las consultas del agente.

In [1]:
# Replace this value with your forwarded URL ending with /api/mcp
CODESPACES_FORWARDED_URL = "https://super-duper-happiness-gjwppqrvwj72wqr-8000.app.github.dev/api/mcp"

### 4. Cargar librerías y variables de entorno para el cliente

In [2]:
import asyncio
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework import ChatAgent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(find_dotenv(usecwd=True))

# Configure the GitHub Models client
MODEL_NAME = os.getenv("GITHUB_MODEL", "openai/gpt-4o")

### 5. Crear y ejecutar el agente cliente

In [3]:
async def query_inventory(prompt, server_url):
    if not server_url or "YOUR_URL_HERE" in server_url:
        print("Error: Please set CODESPACES_FORWARDED_URL with the public server URL.")
        return

    print(f"Connecting to the remote MCP server at {server_url}...")
    
    try:
        client = OpenAIChatClient(
            model_id=MODEL_NAME,
            api_key=os.environ["GITHUB_TOKEN"],
            base_url="https://models.github.ai/inference"
        )
        
        async with (
            MCPStreamableHTTPTool(
                name="inventoryservice",
                url=server_url,
            ) as mcp_server,
            ChatAgent(
                chat_client=client,
                name="InventoryAgent",
                instructions=(
                    "Eres un asistente de gestión de inventario que puede consultar niveles de stock y ventas semanales a través de un servidor remoto."
                ),
            ) as agent,
        ):
            print("Connected to the remote MCP server.")
            print(f"Running inventory query: {prompt}")
            result = await agent.run(prompt, tools=mcp_server)
            print("Query result:")
            print(result)
    except Exception as e:
        print(f"Error while connecting to the MCP server: {type(e).__name__}")
        print(f"Details: {str(e)}")
        print("\nPossible fixes:")
        print("1. Verify the server is running: python remote_mcp_server.py")
        print("2. Ensure the URL is correct, public, and ends with /api/mcp")
        print("3. Try opening the URL in a browser to verify connectivity")

In [4]:
print("Starting inventory analysis with get_inventory_levels...")
user_prompt = "Usa la herramienta get_inventory_levels para mostrar los niveles actuales de inventario."
await query_inventory(user_prompt, CODESPACES_FORWARDED_URL)
print("Inventory analysis complete.")

Starting inventory analysis with get_inventory_levels...
Connecting to the remote MCP server at https://super-duper-happiness-gjwppqrvwj72wqr-8000.app.github.dev/api/mcp...
Connected to the remote MCP server.
Running inventory query: Usa la herramienta get_inventory_levels para mostrar los niveles actuales de inventario.
Connected to the remote MCP server.
Running inventory query: Usa la herramienta get_inventory_levels para mostrar los niveles actuales de inventario.
Query result:
Los niveles actuales de inventario son los siguientes:

- Laptop: 12 unidades
- Monitor: 25 unidades
- Teclado inalámbrico: 40 unidades
- Mouse ergonómico: 34 unidades
- Router WiFi: 18 unidades
- Auriculares: 27 unidades
- Webcam: 15 unidades
- Disco duro externo: 21 unidades
- Silla ergonómica: 10 unidades
- Escritorio ajustable: 8 unidades
Inventory analysis complete.
Query result:
Los niveles actuales de inventario son los siguientes:

- Laptop: 12 unidades
- Monitor: 25 unidades
- Teclado inalámbrico: 4

In [5]:
print("Starting inventory analysis with get_weekly_sales...")
user_prompt = "Usa la herramienta get_weekly_sales para proporcionar el consumo semanal por producto."
await query_inventory(user_prompt, CODESPACES_FORWARDED_URL)
print("Inventory analysis complete.")

Starting inventory analysis with get_weekly_sales...
Connecting to the remote MCP server at https://super-duper-happiness-gjwppqrvwj72wqr-8000.app.github.dev/api/mcp...
Connected to the remote MCP server.
Running inventory query: Usa la herramienta get_weekly_sales para proporcionar el consumo semanal por producto.
Connected to the remote MCP server.
Running inventory query: Usa la herramienta get_weekly_sales para proporcionar el consumo semanal por producto.
Query result:
El consumo semanal por producto es el siguiente:

- Laptop: 9 unidades
- Monitor: 14 unidades
- Teclado inalámbrico: 7 unidades
- Mouse ergonómico: 6 unidades
- Router WiFi: 5 unidades
- Auriculares: 11 unidades
- Webcam: 4 unidades
- Disco duro externo: 8 unidades
- Silla ergonómica: 3 unidades
- Escritorio ajustable: 2 unidades
Inventory analysis complete.
Query result:
El consumo semanal por producto es el siguiente:

- Laptop: 9 unidades
- Monitor: 14 unidades
- Teclado inalámbrico: 7 unidades
- Mouse ergonómico